# Hardware Security Testing

Wintermute is purpose-built for hardware security auditing. This notebook covers:
1. Modeling embedded devices with processors and architectures
2. Debug peripherals — UART and JTAG interfaces
3. Test plans — declarative testing frameworks
4. Test runs — executing and tracking test case status
5. Status reports — operation overview

In [ ]:
from wintermute.core import Operation
from wintermute.hardware import Architecture, Processor

op = Operation("hw-audit-iot-cam")
op.addDevice("iot-camera-v2", "192.168.1.100", os="Linux 5.10")
dev = op.getDeviceByHostname("iot-camera-v2")

print(f"Device: {dev.hostname} ({dev.operatingsystem})")

## Processor and Architecture

The **Processor** dataclass captures the target's CPU details: name, manufacturer, family,
endianness, and an **Architecture** sub-object with core type, instruction set, and features.

In [ ]:
arch = Architecture(
    core="Cortex-A7",
    instruction_set="ARMv7-A",
    cpu_cores=2,
    key_features={"thumb2": True, "neon": True, "trustzone": True},
)
proc = Processor(
    processor="Cortex-A7",
    manufacturer="ARM",
    processor_family="Cortex-A",
    architecture=arch,
    endianness="little",
)
dev.processor = proc

print(f"Processor: {proc.processor} by {proc.manufacturer}")
print(f"Architecture: {arch.instruction_set} ({arch.cpu_cores} cores, {arch.core})")
print(f"Key features: {arch.key_features}")

## Peripherals — UART and JTAG

Hardware debug interfaces are modeled as **Peripheral** subclasses. UART and JTAG are the
most common targets in hardware pentesting — often left enabled in production firmware.

Pins are stored as a dictionary mapping pin names to their physical locations or descriptions.

In [ ]:
from wintermute.peripherals import JTAG, UART

uart = UART(
    name="debug-uart",
    baudrate=115200,
    device_path="/dev/ttyUSB0",
    pins={"tx": "J3-1", "rx": "J3-2", "gnd": "J3-3"},
)
jtag = JTAG(
    name="main-jtag",
    device_path="/dev/jtag0",
    pins={"tck": "J5-1", "tms": "J5-2", "tdi": "J5-3", "tdo": "J5-4", "gnd": "J5-5"},
)
dev.peripherals.extend([uart, jtag])

print(f"Peripherals on {dev.hostname}:")
for p in dev.peripherals:
    print(f"  {p.name} ({type(p).__name__}) — pins: {p.pins}")

## Test Plans

**Test plans** are declarative testing frameworks loaded from JSON. Each plan contains
test cases with steps, execution modes, and target bindings.

Wintermute ships with test plans in `TestPlans/` — here we load one from disk.

In [ ]:
import json
from pathlib import Path

tp_path = Path("../TestPlans/TP-HW-BLACKBOX-001.json")
if tp_path.exists():
    with open(tp_path) as f:
        tp_data = json.load(f)
    print(f"Test Plan: {tp_data.get('name', 'N/A')}")
    print(f"Code: {tp_data.get('code', 'N/A')}")
    print(f"Test Cases: {len(tp_data.get('test_cases', []))}")
    for tc in tp_data.get("test_cases", [])[:3]:
        print(
            f"  - [{tc.get('code')}] {tc.get('name')}: {tc.get('description', '')[:60]}"
        )
else:
    print("TestPlan file not found — run from examples/ directory")

## Adding Test Plans to an Operation

Test plans can also be created programmatically and added to an operation via `addTestPlan()`.
Each plan needs a `code`, `name`, and `description`.

In [ ]:
from wintermute.core import TestCase, TestPlan

tp = TestPlan(
    code="TP-UART-001",
    name="UART Security Check",
    description="Verify UART debug interface security controls",
    test_cases=[
        TestCase(
            code="TC-UART-001",
            name="UART Root Shell Check",
            description="Verify that UART does not provide unauthenticated root shell",
        ),
        TestCase(
            code="TC-UART-002",
            name="UART Baud Rate Discovery",
            description="Attempt auto-detection of baud rate on debug UART",
        ),
    ],
)
op.addTestPlan(tp)

print(f"Test Plans: {[(t.code, t.name) for t in op.test_plans]}")

## Test Runs

**Test case runs** are generated from test plans via `generateTestRuns()`. Each run tracks
execution status, timestamps, findings, and the analyst who performed it.

Statuses: `not_run` → `in_progress` → `passed` / `failed` / `blocked` / `not_applicable`

In [ ]:
from wintermute.core import RunStatus

runs = op.generateTestRuns()
print(f"Generated {len(runs)} test run(s)")
for run in runs:
    print(
        f"  Run {run.run_id[:8]}... | Case: {run.test_case_code} | Status: {run.status.value}"
    )

# Simulate starting a test
if runs:
    runs[0].status = RunStatus.in_progress
    runs[0].start()
    print(f"\nStarted: {runs[0].test_case_code} -> {runs[0].status.value}")
    print(f"  Started at: {runs[0].started_at}")

## Status Reports

Generate a status report for a time window to see test run progress across the operation.

In [ ]:
from datetime import datetime, timedelta, timezone

now = datetime.now(timezone.utc)
report = op.statusReport(start=now - timedelta(days=1), end=now)
print("Status Report:")
print(f"  Total runs: {report.get('total_runs', 0)}")
print(f"  By status: {report.get('by_status', {})}")

## Summary

A complete hardware security audit workflow in Wintermute:
1. **Create an Operation** — the top-level engagement container
2. **Add Devices** — target hardware with processor and architecture details
3. **Map Peripherals** — UART, JTAG, SWD, I2C, SPI debug interfaces
4. **Load Test Plans** — from JSON or build programmatically
5. **Generate Test Runs** — track execution status per test case
6. **Report** — generate status reports for stakeholders